In [1]:
from sklearn.model_selection import KFold
from SIDER_dataset.libraries.XofN_library import *
from SIDER_dataset.libraries.PCT_library import run_PCT, remove_PCT_files
from SIDER_dataset.libraries.utils import get_clus_path
%load_ext autoreload
%autoreload 2

In [2]:
# Set ADR to predict and scoring
clus_path = get_clus_path()
paths = get_dataset_paths(verbose=True)
print(len(paths), "datasets")

Processing fingerprint for 6 labels: ['se_C0027497', 'se_C0018681', 'se_C0011603', 'se_C0015230', 'se_C0042963', 'se_C0012833']
fingerprint_frequent
C:\Users\KostasVoror\Projects/SEP/SIDER_dataset/datasets/clean_multi_label_datasets/fingerprint_frequent.csv 

Processing CPI for 6 labels: ['se_C0027497', 'se_C0018681', 'se_C0011603', 'se_C0015230', 'se_C0042963', 'se_C0012833']
CPI_frequent
C:\Users\KostasVoror\Projects/SEP/SIDER_dataset/datasets/clean_multi_label_datasets/CPI_frequent.csv 

Processing CPI+fingerprint for 6 labels: ['se_C0027497', 'se_C0018681', 'se_C0011603', 'se_C0015230', 'se_C0042963', 'se_C0012833']
CPI+fingerprint_frequent
C:\Users\KostasVoror\Projects/SEP/SIDER_dataset/datasets/clean_multi_label_datasets/CPI+fingerprint_frequent.csv 

Processing fingerprint for 6 labels: ['se_C0027769', 'se_C0020580', 'se_C0014457', 'se_C0017181', 'se_C0151763', 'se_C0038358']
fingerprint_diverse
C:\Users\KostasVoror\Projects/SEP/SIDER_dataset/datasets/clean_multi_label_datasets/

In [3]:
k = 10
random_state = 42
performances = []
ranking_criteria = ["MDI"]
include_original_features_options = [True, False]
training_algorithm = "PCT"
eval_criteria = ["averageAUROC", "HammingLoss", "SubsetAccuracy"]
max_size = 5
cv_results = []
paths = [paths[6]]  # for wrapper do 1 by 1
for idx, path in enumerate(paths, start=1):
    run_config = f"\n--- Running with label:'{path["label_set"]}' training_algorithm:'{training_algorithm}' eval_criterion:'{eval_criteria}' max_size:'{max_size}' ---"
    print(run_config)
    run_config_name = "_".join(
        [
            path["dataset_name"],
            training_algorithm,
            "_".join(eval_criteria),
            str(max_size),
        ]
    )
    logging_path = f"XofN_wrapper/logs/{run_config_name}_logs.txt"
    print(f"Logs can be found in {logging_path}.")
    logger = get_logger(logging_path)
    logger.info(run_config_name)

    # Load dataset
    current_df = pd.read_csv(path["dataset_path"])
    features = get_features(current_df, path["label_set"])
    # current_df = current_df[features[:100] + path["label_set"]]
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    for fold, (train_idx, test_idx) in enumerate(kf.split(current_df), start=1):
        print(f"\nFold {fold}/{k} ({path["dataset_name"]} {idx}/{len(paths)})")
        train_dataset = current_df.iloc[train_idx]
        test_dataset = current_df.iloc[test_idx]

        feature_rankings = calculate_mdi_multi_rf(train_dataset, path["label_set"])

        XofN_groupings, avg_features, gen_XofN_time = generate_XofN_list_multi(
            train_dataset,
            feature_rankings,
            eval_criteria,
            max_size,
            path["label_set"],
            False,
            logger,
            clus_path,
            "XofN_wrapper/tmp"
        )
        remove_PCT_files("XofN_wrapper/tmp")
        if len(XofN_groupings) == 0:
            print("no XofN groupings were created")
        else:
            for include_original_features in include_original_features_options:
                current_train_dataset = group_features(
                    train_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_test_dataset = group_features(
                    test_dataset,
                    path["label_set"],
                    XofN_groupings,
                    include_original_features,
                    verbose=True
                )

                current_train_dataset.to_csv(f"XofN_wrapper/tmp/train_dataset.csv", index=False)
                current_test_dataset.to_csv(f"XofN_wrapper/tmp/test_dataset.csv", index=False)

                original_res, pruned_res, training_time = run_PCT(clus_path,
                                                                  "XofN_wrapper/tmp/train_dataset.csv",
                                                                  path["label_set"],
                                                                  eval_criteria,
                                                                  test_dataset_path=f"XofN_wrapper/tmp/test_dataset.csv")
                pruned_performance = get_fold_results(pruned_res, eval_criteria, True, fold, include_original_features,
                                                      XofN_groupings,
                                                      gen_XofN_time,
                                                      training_time, path["dataset_name"])
                performances.append(pruned_performance)
                performance = get_fold_results(original_res, eval_criteria, False, fold, include_original_features,
                                               XofN_groupings,
                                               gen_XofN_time,
                                               training_time, path["dataset_name"])
                performances.append(performance)

    if len(performances) == 0:
        print("no XofN groupings were created in any fold")
    else:
        final_perf_df = pd.DataFrame(performances)
        averages = final_perf_df.groupby(["pruning", 'include_original_features', 'dataset'])[
            ['averageAUROC', 'HammingLoss', 'SubsetAccuracy', 'nodes', 'leaves', 'groups',
             'avg_group_features', 'gen_XofN_time', 'training_time']].mean().reset_index()
        print(averages)
        (cv_results.append

         (averages))
        performances = []
# paths[0] - features[:10]
# laptop ??m
# desktop 1.34m


--- Running with label:'['se_C0027497', 'se_C0018681', 'se_C0011603', 'se_C0015230', 'se_C0042963', 'se_C0012833', 'se_C0027769', 'se_C0020580', 'se_C0014457', 'se_C0017181', 'se_C0151763', 'se_C0038358']' training_algorithm:'PCT' eval_criterion:'['averageAUROC', 'HammingLoss', 'SubsetAccuracy']' max_size:'5' ---
Logs can be found in XofN_wrapper/logs/fingerprint_all_PCT_averageAUROC_HammingLoss_SubsetAccuracy_5_logs.txt.

Fold 1/10 (fingerprint_all 1/1)

generate_XofN_list -> Generating groupings based on eval_criterion:['averageAUROC', 'HammingLoss', 'SubsetAccuracy'].


🔄 Processing features: 100%|██████████| 540/540 [1:58:14<00:00, 13.14s/feat]  


XofN_groups: [['f_374', 'f_398', 'f_20', 'f_715', 'f_284'], ['f_23', 'f_455', 'f_285', 'f_14', 'f_543'], ['f_308', 'f_132', 'f_635', 'f_344', 'f_406'], ['f_346', 'f_343', 'f_752', 'f_333', 'f_441'], ['f_366', 'f_706', 'f_451', 'f_393', 'f_15'], ['f_287', 'f_401', 'f_351', 'f_10', 'f_522'], ['f_2', 'f_413', 'f_443', 'f_420', 'f_332'], ['f_19', 'f_129', 'f_613', 'f_345', 'f_178'], ['f_299', 'f_131', 'f_338', 'f_707', 'f_460'], ['f_571', 'f_422', 'f_815', 'f_696', 'f_143'], ['f_697', 'f_483', 'f_464', 'f_861', 'f_26'], ['f_24', 'f_532', 'f_390', 'f_778', 'f_438'], ['f_452', 'f_216', 'f_9', 'f_768', 'f_440'], ['f_391', 'f_515', 'f_392', 'f_446', 'f_146'], ['f_335', 'f_214', 'f_12', 'f_704', 'f_845'], ['f_339', 'f_625', 'f_535', 'f_579', 'f_729'], ['f_639', 'f_840', 'f_249', 'f_416', 'f_503'], ['f_672', 'f_551', 'f_11', 'f_118', 'f_725'], ['f_380', 'f_511', 'f_0', 'f_388'], ['f_186', 'f_466', 'f_430', 'f_654', 'f_711'], ['f_528', 'f_395', 'f_569', 'f_507', 'f_200'], ['f_656', 'f_394', 'f_21

🔄 Processing features: 100%|██████████| 540/540 [1:57:52<00:00, 13.10s/feat]  


XofN_groups: [['f_374', 'f_489', 'f_340', 'f_390', 'f_580'], ['f_20', 'f_511', 'f_592', 'f_14', 'f_375'], ['f_308', 'f_631', 'f_449', 'f_376', 'f_401'], ['f_366', 'f_860', 'f_43', 'f_0', 'f_333'], ['f_346', 'f_242', 'f_10', 'f_151', 'f_413'], ['f_406', 'f_763', 'f_660', 'f_11', 'f_131'], ['f_571', 'f_839', 'f_670', 'f_214', 'f_480'], ['f_2', 'f_129', 'f_570', 'f_613', 'f_540'], ['f_23', 'f_16', 'f_420', 'f_284', 'f_15'], ['f_287', 'f_399', 'f_683', 'f_300', 'f_408'], ['f_338', 'f_395', 'f_442', 'f_216', 'f_625'], ['f_186', 'f_812', 'f_143', 'f_684', 'f_508'], ['f_656', 'f_132', 'f_645', 'f_588', 'f_461'], ['f_392', 'f_215', 'f_146', 'f_412', 'f_451'], ['f_617', 'f_466', 'f_556', 'f_749', 'f_202'], ['f_335', 'f_361', 'f_672', 'f_241', 'f_699'], ['f_391', 'f_25', 'f_355', 'f_611', 'f_344'], ['f_452', 'f_467', 'f_560', 'f_285', 'f_365'], ['f_299', 'f_227', 'f_255', 'f_437', 'f_33'], ['f_12', 'f_622', 'f_696', 'f_341', 'f_474'], ['f_192', 'f_861', 'f_178', 'f_339', 'f_780'], ['f_643', 'f_5

...

XofN_groups: [['f_374', 'f_467', 'f_19', 'f_420', 'f_375'], ['f_20', 'f_300', 'f_390', 'f_0'], ['f_346', 'f_419', 'f_522', 'f_11', 'f_517'], ['f_308', 'f_361', 'f_284', 'f_366', 'f_719'], ['f_2', 'f_131', 'f_229', 'f_299', 'f_15'], ['f_571', 'f_515', 'f_333', 'f_473', 'f_510'], ['f_406', 'f_752', 'f_178', 'f_441', 'f_186'], ['f_23', 'f_343', 'f_195'], ['f_672', 'f_551', 'f_582', 'f_689', 'f_533'], ['f_566', 'f_454', 'f_442', 'f_631', 'f_656'], ['f_639', 'f_408', 'f_528', 'f_749', 'f_535'], ['f_338', 'f_216', 'f_146', 'f_368', 'f_213'], ['f_185', 'f_394', 'f_143', 'f_696', 'f_734'], ['f_339', 'f_644', 'f_613', 'f_558', 'f_116'], ['f_287', 'f_588', 'f_345', 'f_336', 'f_629'], ['f_391', 'f_514', 'f_445', 'f_115', 'f_593'], ['f_452', 'f_505', 'f_443', 'f_681', 'f_527'], ['f_3', 'f_494', 'f_553', 'f_653', 'f_645'], ['f_12', 'f_398', 'f_697', 'f_789', 'f_635'], ['f_516', 'f_839', 'f_840', 'f_569', 'f_717'], ['f_643', 'f_401', 'f_451', 'f_145', 'f_715'], ['f_335', 'f_489', 'f_684', 'f_646', '

🔄 Processing features: 100%|██████████| 540/540 [2:08:49<00:00, 14.31s/feat]  


XofN_groups: [['f_374', 'f_401', 'f_356', 'f_664', 'f_643'], ['f_20', 'f_588', 'f_299', 'f_619', 'f_213'], ['f_23', 'f_284', 'f_15', 'f_697', 'f_438'], ['f_287', 'f_661', 'f_679', 'f_406', 'f_567'], ['f_308', 'f_587', 'f_178', 'f_510', 'f_544'], ['f_346', 'f_394', 'f_710', 'f_333', 'f_298'], ['f_2', 'f_494', 'f_443', 'f_455', 'f_526'], ['f_338', 'f_515', 'f_709', 'f_749', 'f_728'], ['f_672', 'f_131', 'f_430', 'f_475', 'f_200'], ['f_393', 'f_582', 'f_285', 'f_1', 'f_497'], ['f_516', 'f_467', 'f_391', 'f_340', 'f_26'], ['f_366', 'f_129', 'f_351', 'f_14', 'f_646'], ['f_571', 'f_359', 'f_255', 'f_439', 'f_398'], ['f_656', 'f_132', 'f_615', 'f_18', 'f_658'], ['f_392', 'f_300', 'f_384', 'f_592', 'f_824'], ['f_380', 'f_767', 'f_9', 'f_698', 'f_532'], ['f_614', 'f_35', 'f_708', 'f_640', 'f_151'], ['f_566', 'f_644', 'f_613', 'f_683', 'f_483'], ['f_440', 'f_423', 'f_420', 'f_335', 'f_543'], ['f_617', 'f_454', 'f_688', 'f_371', 'f_449'], ['f_24', 'f_244', 'f_376', 'f_451', 'f_10'], ['f_19', 'f_61

🔄 Processing features: 100%|██████████| 540/540 [2:12:38<00:00, 14.74s/feat]  


XofN_groups: [['f_374', 'f_749', 'f_713', 'f_255', 'f_244'], ['f_20', 'f_515', 'f_355', 'f_351', 'f_230'], ['f_308', 'f_706', 'f_441', 'f_406', 'f_667'], ['f_287', 'f_778', 'f_356', 'f_15', 'f_653'], ['f_346', 'f_860', 'f_9', 'f_43', 'f_332'], ['f_23', 'f_715', 'f_430', 'f_421', 'f_292'], ['f_393', 'f_131', 'f_575', 'f_708', 'f_11'], ['f_366', 'f_242', 'f_10', 'f_420', 'f_443'], ['f_2', 'f_812', 'f_384', 'f_628', 'f_370'], ['f_186', 'f_616', 'f_143', 'f_845', 'f_283'], ['f_338', 'f_861', 'f_697', 'f_800', 'f_241'], ['f_299', 'f_588', 'f_642', 'f_416', 'f_129'], ['f_672', 'f_839', 'f_1', 'f_833', 'f_553'], ['f_566', 'f_216', 'f_146', 'f_368', 'f_493'], ['f_614', 'f_840', 'f_285', 'f_452', 'f_390'], ['f_391', 'f_489', 'f_625', 'f_668', 'f_635'], ['f_571', 'f_359', 'f_696', 'f_506', 'f_347'], ['f_617', 'f_436', 'f_178', 'f_581', 'f_361'], ['f_19', 'f_728', 'f_464', 'f_394', 'f_629'], ['f_335', 'f_742', 'f_704', 'f_30', 'f_522'], ['f_12', 'f_343', 'f_249', 'f_450', 'f_712'], ['f_339', 'f_6

In [4]:
# paths[6] - fingerprint_diverse
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,False,no_org,fingerprint_all,0.565048,0.279389,0.054962,980.6,490.8,109.6,4.677277,7405.778034,0.806474
1,False,with_org,fingerprint_all,0.567379,0.275448,0.062595,972.0,486.5,109.6,4.677277,7405.778034,1.433403
2,True,no_org,fingerprint_all,0.503060,0.200827,0.105343,2.2,1.6,109.6,4.677277,7405.778034,0.806474
3,True,with_org,fingerprint_all,0.513947,0.199363,0.109923,3.6,2.3,109.6,4.677277,7405.778034,1.433403


In [22]:
# paths[7] - fingerprint_diverse
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,False,no_org,fingerprint_diverse,0.577506,0.273612,0.239396,786.4,393.7,105.5,4.696913,6158.804027,0.787095
1,False,with_org,fingerprint_diverse,0.579968,0.261364,0.254546,787.4,394.2,105.5,4.696913,6158.804027,1.414068
2,True,no_org,fingerprint_diverse,0.511022,0.180682,0.431818,5.0,3.0,105.5,4.696913,6158.804027,0.787095
3,True,with_org,fingerprint_diverse,0.524099,0.178913,0.435607,6.6,3.8,105.5,4.696913,6158.804027,1.414068


In [5]:
# paths[8] - fingerprint_frequent
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,False,no_org,fingerprint_frequent,0.570636,0.285095,0.350319,602.0,301.5,84.0,4.863861,6396.208116,0.852198
1,False,with_org,fingerprint_frequent,0.568670,0.277887,0.360173,596.4,298.7,84.0,4.863861,6396.208116,1.270581
2,True,no_org,fingerprint_frequent,0.559994,0.232903,0.503828,51.8,26.4,84.0,4.863861,6396.208116,0.852198
3,True,with_org,fingerprint_frequent,0.558241,0.230876,0.501540,51.4,26.2,84.0,4.863861,6396.208116,1.270581


In [5]:
# All results
save_path = "XofN_wrapper/"
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res.sort_values(by='dataset', ascending=False, inplace=True)
final_grouped_res = final_grouped_res.reset_index(drop=True)

if os.path.exists(save_path + "all_results.csv"):
    existing = pd.read_csv(save_path + "all_results.csv", index_col=0)
    combined = pd.concat([existing, final_grouped_res], ignore_index=True)
    combined.drop_duplicates(inplace=True)
else:
    combined = final_grouped_res
combined.to_csv(save_path + "all_results.csv")
combined

,pruning,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time
0,False,no_org,fingerprint_frequent,0.570636,0.285095,0.350319,602.0,301.5,84.0,4.863861,6396.208116,0.852198
1,False,with_org,fingerprint_frequent,0.568670,0.277887,0.360173,596.4,298.7,84.0,4.863861,6396.208116,1.270581
2,True,no_org,fingerprint_frequent,0.559994,0.232903,0.503828,51.8,26.4,84.0,4.863861,6396.208116,0.852198
3,True,with_org,fingerprint_frequent,0.558241,0.230876,0.501540,51.4,26.2,84.0,4.863861,6396.208116,1.270581
4,False,no_org,fingerprint_diverse,0.577506,0.273612,0.239396,786.4,393.7,105.5,4.696913,6158.804027,0.787095
5,False,with_org,fingerprint_diverse,0.579968,0.261364,0.254546,787.4,394.2,105.5,4.696913,6158.804027,1.414068
6,True,no_org,fingerprint_diverse,0.511022,0.180682,0.431818,5.0,3.0,105.5,4.696913,6158.804027,0.787095
7,True,with_org,fingerprint_diverse,0.524099,0.178913,0.435607,6.6,3.8,105.5,4.696913,6158.804027,1.414068
8,False,no_org,fingerprint_all,0.565048,0.279389,0.054962,980.6,490.8,109.6,4.677277,7405.778034,0.806474
9,False,with_org,fingerprint_all,0.567379,0.275448,0.062595,972.0,486.5,109.6,4.677277,7405.778034,1.433403


In [6]:
# Table ready (with pruning, rounded, compact)
save_path = "XofN_wrapper/"
res = pd.read_csv(save_path + "all_results.csv", index_col=0)
table_results = get_table_results(res, get_dataset_paths())
table_results.to_csv(save_path + "table_results.csv")
table_results

,include_original_features,dataset,averageAUROC,HammingLoss,SubsetAccuracy,Nodes; Leaves,#XofN; #Feat/XofN,# Ung. Feats,XofN time (s); PCT tr. time (s)
2,no_org,fingerprint_frequent,0.560,0.233,0.504,51.8; 26.4,84.0; 4.9,131.435655,6396.2; 0.9
3,with_org,fingerprint_frequent,0.558,0.231,0.502,51.4; 26.2,84.0; 4.9,131.435655,6396.2; 1.3
6,no_org,fingerprint_diverse,0.511,0.181,0.432,5.0; 3.0,105.5; 4.7,44.475715,6158.8; 0.8
7,with_org,fingerprint_diverse,0.524,0.179,0.436,6.6; 3.8,105.5; 4.7,44.475715,6158.8; 1.4
10,no_org,fingerprint_all,0.503,0.201,0.105,2.2; 1.6,109.6; 4.7,27.370465,7405.8; 0.8
11,with_org,fingerprint_all,0.514,0.199,0.110,3.6; 2.3,109.6; 4.7,27.370465,7405.8; 1.4


In [9]:
# Frequent with time
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res

,pruning,include_original_features,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time,dataset
0,False,no_org,0.568567,0.274587,0.389788,599.6,300.3,18.6,2.417036,5984.643825,1.109454,fingerprint_frequent
1,False,with_org,0.571161,0.280927,0.377649,598.8,299.9,18.6,2.417036,5984.643825,1.198192,fingerprint_frequent
2,True,no_org,0.518590,0.226579,0.520557,19.8,10.4,18.6,2.417036,5984.643825,1.109454,fingerprint_frequent
3,True,with_org,0.530389,0.229251,0.515213,27.6,14.3,18.6,2.417036,5984.643825,1.198192,fingerprint_frequent


In [4]:
# Diverse with time
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res

,pruning,include_original_features,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time,dataset
0,False,no_org,0.584899,0.270961,0.230303,791.6,396.3,67.9,2.527317,5860.982623,1.114352,fingerprint_diverse
1,False,with_org,0.583008,0.271718,0.237121,803.0,402.0,67.9,2.527317,5860.982623,1.280688,fingerprint_diverse
2,True,no_org,0.515306,0.179167,0.438638,5.2,3.1,67.9,2.527317,5860.982623,1.114352,fingerprint_diverse
3,True,with_org,0.518883,0.178788,0.437880,5.0,3.0,67.9,2.527317,5860.982623,1.280688,fingerprint_diverse


In [4]:
# All with time
final_grouped_res = pd.DataFrame()
for df in cv_results:
    final_grouped_res = pd.concat([final_grouped_res, df])
final_grouped_res

,pruning,include_original_features,averageAUROC,HammingLoss,SubsetAccuracy,nodes,leaves,groups,avg_group_features,gen_XofN_time,training_time,dataset
0,False,no_org,0.586784,0.266476,0.069466,970.2,485.6,75.4,2.35369,6578.325103,1.162779,fingerprint_all
1,False,with_org,0.592362,0.264057,0.070229,971.6,486.3,75.4,2.35369,6578.325103,1.355200,fingerprint_all
2,True,no_org,0.515989,0.198537,0.111450,3.2,2.1,75.4,2.35369,6578.325103,1.162779,fingerprint_all
3,True,with_org,0.515989,0.198537,0.111450,3.2,2.1,75.4,2.35369,6578.325103,1.355200,fingerprint_all
